In [1]:
import torch
folder = '283_231_250_0/'
tens = torch.load(folder + '0.pt')
tens.shape

torch.Size([8, 4, 32, 32])

In [ ]:
import os
import torch
import numpy as np
from scipy import linalg

# ── helpers ──────────────────────────────────────────────────────────────────

def load_tensors_from_folder(folder: str) -> np.ndarray:
    """
    Loads all .pt files from a folder.
    Each file is expected to contain a tensor of shape (B, 4, 32, 32).
    Returns a numpy array of shape (N, 4096) — each sample flattened.
    """
    all_tensors = []
    files = sorted(os.listdir(folder), key=lambda x: int(x.split('.')[0]))
    for fname in files:
        if not fname.endswith('.pt'):
            continue
        path = os.path.join(folder, fname)
        t = torch.load(path, map_location='cpu')   # shape: (B, 4, 32, 32)
        t = t.float().view(t.shape[0], -1)         # shape: (B, 4096)
        all_tensors.append(t.numpy())
    data = np.concatenate(all_tensors, axis=0)     # shape: (N, 4096)
    print(f"Loaded {data.shape[0]} samples from '{folder}'  shape: {data.shape}")
    return data


def compute_stats(features: np.ndarray):
    """Compute mean and covariance of a feature matrix."""
    mu = np.mean(features, axis=0)
    sigma = np.cov(features, rowvar=False)
    return mu, sigma


def frechet_distance(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    """
    Fréchet distance between two multivariate Gaussians:
        FID = ||mu1 - mu2||^2 + Tr(sigma1 + sigma2 - 2 * sqrt(sigma1 @ sigma2))
    """
    diff = mu1 - mu2
    # Matrix square root via eigendecomposition (more stable than sqrtm for large matrices)
    covmean, _ = linalg.sqrtm(sigma1 @ sigma2, disp=False)

    # Handle numerical issues → imaginary components
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError(f"Imaginary component too large: {m}")
        covmean = covmean.real

    # Regularise if matrix is nearly singular
    if not np.isfinite(covmean).all():
        print("Warning: singular product; adding epsilon to diagonal of covariances")
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset) @ (sigma2 + offset))

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return float(fid.real)

folder_real = "283/"                  # batch_size=32 files
folder_fake = "283_231_250_0/"        # batch_size=8 files

feats_real = load_tensors_from_folder(folder_real)
feats_fake = load_tensors_from_folder(folder_fake)

mu1, sigma1 = compute_stats(feats_real)
mu2, sigma2 = compute_stats(feats_fake)

fid_score = frechet_distance(mu1, sigma1, mu2, sigma2)
print(f"\nLatent FID: {fid_score:.4f}")
